### [문제] bmi.csv 데이터를 사용하여 네이티브 케라스로 분류하는 모델을 작성하여 정확도를 출력하는 코드를 작성하시오

제출형식: 홍길동_0521_정확도.ipynb

<처리조건>  
1. 성능이 최대가 되도록 최선을 다할것!!!
2. 훈련데이터와 테스터 데이터 비율은 9:1로 한다.

In [1]:
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Flatten
import np_utils                                                   # pip install np_utils
from keras.utils import to_categorical
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
data = pd.read_csv("bmi.csv")
data.head(5)

,height,weight,label
0,142,62,fat
1,142,73,fat
2,177,61,normal
3,187,48,thin
4,153,60,fat


In [3]:
# 종속변수 확인
data['label'].unique()

<StringArray>
['fat', 'normal', 'thin']
Length: 3, dtype: str

In [4]:
# 독립변수와 종속변수 분리
xdata = data[['height', 'weight']]
tdata = pd.DataFrame(data['label'])

In [5]:
tdata_one = pd.get_dummies(data['label'], dtype=int)
print(tdata_one.head())

   fat  normal  thin
0    1       0     0
1    1       0     0
2    0       1     0
3    0       0     1
4    1       0     0


In [6]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()

xdata_ss = sc.fit_transform(xdata) 
print(xdata_ss.shape, tdata_one.shape)

(20000, 2) (20000, 3)


In [7]:
# 9대 1로 비율 나누기

xtrain = xdata_ss[:18000, :]
xtest = xdata_ss[18000:, :]

# xdata_ss는 위 셀에서 넘파이 배열로 바뀌는 과정인 'xdata_ss = sc.fit_transform(xdata)'를 거쳤다
# 하지만 종속변수인 ttest는 그러지 못해서 현재 '판다스 데이터프레임'형태이다. 그렇기에
# 판다스의 슬라이싱 함수인 iloc를 사용해서 쉽표를 기준으로 앞은 행 뒤는 열을 지정해 정수를 가져온다.
ttrain = tdata_one.iloc[:18000, :]
ttest = tdata_one.iloc[18000:, :]

# 학습데이터와 검증 데이터의 구조 확인용
print( xtrain.shape, ttrain.shape, xtest.shape, ttest.shape)

(18000, 2) (18000, 3) (2000, 2) (2000, 3)


In [8]:
model = Sequential()

# 1번째 은닉층
model.add(Flatten(input_shape=(2,)))      # input_shape(,) # keras의 input_shape은 튜플 타입으로 정보를 받기에 ,로 튜플로 인식하게 해줌
model.add(Dense(512, activation='relu'))  
model.add(Dropout(0.2))

# 2번째 은닉층
model.add(Dense(512, activation='relu'))  
model.add(Dropout(0.2))

# 출력층
model.add(Dense(3, activation='softmax'))  
model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten (Flatten)           (None, 2)                 0         
                                                                 
 dense (Dense)               (None, 512)               1536      
                                                                 
 dropout (Dropout)           (None, 512)               0         
                                                                 
 dense_1 (Dense)             (None, 512)               262656    
                                                                 
 dropout_1 (Dropout)         (None, 512)               0         
                                                                 
 dense_2 (Dense)             (None, 3)                 1539      
                                                                 
Total params: 265731 (1.01 MB)
Trainable params: 265731

In [9]:
model.compile(
    loss='categorical_crossentropy',   # 다중분류, categorical_crossentropy => 타겟이 원핫 인코딩 일때  //  sparse_categorical_crossentropy = > 정답이 정수일때 
    optimizer='adam',
    metrics=['accuracy'])

In [10]:
hist = model.fit(xtrain, ttrain, epochs=5, batch_size=32)

Epoch 1/5


563/563 [==============================] - 4s 5ms/step - loss: 0.0985 - accuracy: 0.9613
Epoch 2/5
563/563 [==============================] - 2s 4ms/step - loss: 0.0442 - accuracy: 0.9817
Epoch 3/5
563/563 [==============================] - 2s 4ms/step - loss: 0.0328 - accuracy: 0.9857
Epoch 4/5
563/563 [==============================] - 2s 4ms/step - loss: 0.0332 - accuracy: 0.9864
Epoch 5/5
563/563 [==============================] - 2s 4ms/step - loss: 0.0303 - accuracy: 0.9871


In [11]:
score = model.evaluate(xtest, ttest, verbose=1)
score

63/63 [==============================] - 0s 971us/step - loss: 0.0204 - accuracy: 0.9940


[0.020404592156410217, 0.9940000176429749]